# s1figure.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('wod_monthly_mld_file')
mld_true = xr.open_mfdataset(fdir1)
#mld_true = mld_true.sel(time=slice('1982-1','2023-8'))
mld_true

In [ ]:
count_valid = mld_true.notnull().sum(dim='time')

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker

# =========================
# 投影：Robinson
# =========================
proj = ccrs.Robinson(central_longitude=180)

fig = plt.figure(figsize=(8, 4.5))
ax = plt.axes(projection=proj)

# 数据
plot_data = count_valid.MLD_d003.where(mask_ocean)

# =========================
# 等值面图
# =========================
im = plot_data.plot.contourf(
    ax=ax,
    transform=ccrs.PlateCarree(),   # 数据仍是经纬度
    cmap=plt.cm.Greens,             # 浅绿 → 深绿
    levels=11,
    vmin=0,
    vmax=450,
    add_colorbar=True,
    cbar_kwargs={
        'label': 'Number of valid MLD values',
        'shrink': 0.7,
        'aspect': 25,
        'pad': 0.02,
    }
)

# =========================
# 标题
# =========================
ax.set_title(
    'Valid MLD sample count at each grid point',
    loc='center',
    fontsize=16
)

# =========================
# 地理要素
# =========================
ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)

# Robinson 必须 global
ax.set_global()

# =========================
# 网格线
# =========================
gl = ax.gridlines(
    draw_labels=True,
    dms=True,
    x_inline=False,
    y_inline=False
)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = ticker.FixedLocator(range(-180, 181, 60))
gl.ylocator = ticker.FixedLocator(range(-90, 91, 30))
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

# =========================
# 子图标注
# =========================
ax.text(
    0.02, 0.95, '(a)',
    transform=ax.transAxes,
    fontsize=14,
    fontweight='bold',
    va='top',
    ha='left',
    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none')
)

plt.tight_layout()
plt.show()


In [ ]:
yearly_valid_sample_count = (
    mld_true
    .where(mask_ocean)
    .notnull()
    .sum(dim=('lat', 'lon'))   # 先变成 (time,)
    .groupby('time.year')
    .sum(dim='time')           # 再按年加
)
yearly_valid_sample_count

In [ ]:
years = yearly_valid_sample_count.coords['year'].values
values = yearly_valid_sample_count.MLD_d003.values

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.figure(figsize=(8, 4))

plt.plot(
    years,
    values,
    color='green',        # 颜色改为绿色
    linewidth=2,
    marker='o',
    markersize=4
)

plt.xlabel('Year', fontsize=12)
plt.ylabel('Annual data count', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)

# 设置纵坐标科学计数法格式
ax = plt.gca()
ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker
import matplotlib.ticker as mticker
from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.delaxes(axes[0])
ax1 = fig.add_subplot(1, 2, 1, projection=ccrs.Robinson(central_longitude=180))
ax2 = axes[1]

plot_data = count_valid.MLD_d003.where(mask_ocean)

divider = make_axes_locatable(ax1)
cax = divider.append_axes("bottom", size="5%", pad=0.25, axes_class=plt.Axes)  # 这里传axes_class

im = plot_data.plot.contourf(
    ax=ax1,
    transform=ccrs.PlateCarree(),
    cmap=plt.cm.Greens,
    levels=11,
    vmin=0,
    vmax=450,
    add_colorbar=False,
)

cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
cbar.set_label('Number of valid MLD values')

ax1.set_title('Valid MLD sample count at each grid point', loc='center', fontsize=16)
ax1.coastlines(linewidth=0.8)
ax1.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
ax1.set_global()

gl = ax1.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = ticker.FixedLocator(range(-180, 181, 60))
gl.ylocator = ticker.FixedLocator(range(-90, 91, 30))
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

ax1.text(0.02, 0.95, '(a)', transform=ax1.transAxes,
         fontsize=14, fontweight='bold', va='top', ha='left',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

ax2.plot(
    years,
    values,
    color='green',
    linewidth=2,
    marker='o',
    markersize=4
)

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Annual data count', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.5)

ax2.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax2.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))

ax2.text(0.02, 0.95, '(b)', transform=ax2.transAxes,
         fontsize=14, fontweight='bold', va='top', ha='left',
         bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# 调整两图大小一致
ax1_pos = ax1.get_position()
ax2_pos = ax2.get_position()
new_height = min(ax1_pos.height, ax2_pos.height)
new_y0 = max(ax1_pos.y0, ax2_pos.y0)
ax1.set_position([ax1_pos.x0, new_y0, ax1_pos.width, new_height])
ax2.set_position([ax2_pos.x0, new_y0, ax2_pos.width, new_height])

plt.tight_layout()
plt.show()


# 计算点的数量

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from tqdm import tqdm

years = np.arange(1982, 2024)
months = np.arange(1, 13)

lon_grid = np.arange(-179.5, 180, 1)
lat_grid = np.arange(-89.5, 90, 1)

count_mld_list = []
count_all_list = []
time_list = []

total_months = len(years) * len(months)

with tqdm(total=total_months, desc="Processing profile counts") as pbar:
    for y in years:
        for m in months:
            fpath = f"{configured_path('wod_profile_dir')}/MLD_WOD_griddata_{y:04d}_{m:02d}.nc"
            try:
                ds = xr.open_dataset(fpath)
            except FileNotFoundError:
                tqdm.write(f"缺少文件: {fpath}")
                pbar.update(1)
                continue

            lon = ds["longitude"].values
            lat = ds["latitude"].values
            mld = ds["MLD_d003"].values

            if len(lon) == 0:
                tqdm.write(f"该月没有数据: {fpath}")
                pbar.update(1)
                continue

            df = pd.DataFrame({"lon": lon, "lat": lat, "mld": mld})

            # 映射到网格中心
            df["lon_bin"] = np.floor(df["lon"]) + 0.5
            df["lat_bin"] = np.floor(df["lat"]) + 0.5

            # 所有剖面数统计（无论mld是否NaN）
            df_count_all = df.groupby(["lat_bin", "lon_bin"]).size().reset_index(name="count_all")

            # 能算出MLD的剖面数（mld非NaN）
            df_count_mld = df[df["mld"].notna()].groupby(["lat_bin", "lon_bin"]).size().reset_index(name="count_mld")

            # 透视表，重建网格
            pivot_count_all = df_count_all.pivot(index="lat_bin", columns="lon_bin", values="count_all")
            pivot_count_all = pivot_count_all.reindex(index=lat_grid, columns=lon_grid).fillna(0)

            pivot_count_mld = df_count_mld.pivot(index="lat_bin", columns="lon_bin", values="count_mld")
            pivot_count_mld = pivot_count_mld.reindex(index=lat_grid, columns=lon_grid).fillna(0)

            count_all_grid = xr.DataArray(pivot_count_all.values, coords=[lat_grid, lon_grid], dims=["lat", "lon"])
            count_mld_grid = xr.DataArray(pivot_count_mld.values, coords=[lat_grid, lon_grid], dims=["lat", "lon"])

            count_all_list.append(count_all_grid)
            count_mld_list.append(count_mld_grid)
            time_list.append(np.datetime64(f"{y}-{m:02d}-15"))

            pbar.update(1)

count_all_3d = xr.concat(count_all_list, dim="time").assign_coords(time=("time", time_list))
count_all_3d.name = "profile_count_all"

count_mld_3d = xr.concat(count_mld_list, dim="time").assign_coords(time=("time", time_list))
count_mld_3d.name = "profile_count_mld"

ds_grid = xr.Dataset({
    "profile_count_all": count_all_3d,
    "profile_count_mld": count_mld_3d
})

ds_grid["lat"].attrs = {"units": "degrees_north", "long_name": "latitude"}
ds_grid["lon"].attrs = {"units": "degrees_east", "long_name": "longitude"}
ds_grid["time"].attrs = {"long_name": "time"}

ds_grid["profile_count_all"].attrs = {"units": "count", "long_name": "Total number of profiles in grid cell"}
ds_grid["profile_count_mld"].attrs = {"units": "count", "long_name": "Number of profiles with valid MLD in grid cell"}

encoding = {
    "time": {
        "units": "days since 1900-01-01 00:00:00",
        "calendar": "gregorian"
    }
}

ds_grid.to_netcdf(
    str(Path(configured_path('work_dir')) / 'MLD_profile_counts_1982_2023.nc'),
    encoding=encoding
)


In [ ]:
ds_grid['lon'] = (ds_grid['lon'] + 360) % 360
ds_grid = ds_grid.sortby('lon')
ds_grid

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker

# =========================
# 投影：Robinson
# =========================
proj = ccrs.Robinson(central_longitude=180)

fig = plt.figure(figsize=(8, 4.5))
ax = plt.axes(projection=proj)

# 数据
plot_data = ds_grid.profile_count_all.where(mask_ocean).sum(dim='time')

# =========================
# 等值面图
# =========================
im = plot_data.plot.contourf(
    ax=ax,
    transform=ccrs.PlateCarree(),   # 数据仍是经纬度
    cmap=plt.cm.Greens,             # 浅绿 → 深绿
    levels=11,
    vmin=0,
    vmax=1000,
    add_colorbar=True,
    cbar_kwargs={
        'label': 'Number of valid MLD values',
        'shrink': 0.7,
        'aspect': 25,
        'pad': 0.02,
    }
)

# =========================
# 标题
# =========================
ax.set_title(
    'Valid MLD sample count at each grid point',
    loc='center',
    fontsize=16
)

# =========================
# 地理要素
# =========================
ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)

# Robinson 必须 global
ax.set_global()

# =========================
# 网格线
# =========================
gl = ax.gridlines(
    draw_labels=True,
    dms=True,
    x_inline=False,
    y_inline=False
)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = ticker.FixedLocator(range(-180, 181, 60))
gl.ylocator = ticker.FixedLocator(range(-90, 91, 30))
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

# =========================
# 子图标注
# =========================
ax.text(
    0.02, 0.95, '(a)',
    transform=ax.transAxes,
    fontsize=14,
    fontweight='bold',
    va='top',
    ha='left',
    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none')
)

plt.tight_layout()
plt.show()

In [ ]:
yearly_valid_sample_count = (
    ds_grid.profile_count_all
    .where(mask_ocean)
    .sum(dim=('lat', 'lon'))   # 先变成 (time,)
    .groupby('time.year')
    .sum(dim='time')           # 再按年加
)
yearly_valid_sample_count

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import xarray as xr

# 注册并验证项目内的 Times New Roman 字体
times_font_dir = Path(configured_path('font_dir'))
times_font_files = [
    times_font_dir / 'Times New Roman.ttf',
    times_font_dir / 'Times New Roman Bold.ttf',
]
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.cal': 'Times New Roman:italic',
    'mathtext.sf': 'Times New Roman',
    'mathtext.tt': 'Times New Roman',
})

# 你的数据
plot_data1 = count_valid.MLD_d003.where(mask_ocean)
plot_data2 = ds_grid.profile_count_all.where(mask_ocean).sum(dim='time')
plot_data3 = ds_grid.profile_count_mld.where(mask_ocean).sum(dim='time')

yearly1 = (
    mld_true
    .where(mask_ocean)
    .notnull()
    .sum(dim=('lat', 'lon'))
    .groupby('time.year')
    .sum(dim='time')
)
years1 = yearly1.coords['year'].values
values1 = yearly1.MLD_d003.values

yearly2 = (
    ds_grid.profile_count_all
    .where(mask_ocean)
    .sum(dim=('lat', 'lon'))
    .groupby('time.year')
    .sum(dim='time')
)
years2 = yearly2.coords['year'].values
values2 = yearly2.values

yearly3 = (
    ds_grid.profile_count_mld
    .where(mask_ocean)
    .sum(dim=('lat', 'lon'))
    .groupby('time.year')
    .sum(dim='time')
)
years3 = yearly3.coords['year'].values
values3 = yearly3.values

# 仅保留 a-d 四个子图；地图列稍宽，使左右两列的视觉尺寸更协调
fig = plt.figure(figsize=(17, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], hspace=0.32, wspace=0.06)

map_proj = ccrs.Robinson(central_longitude=210)
ax1 = fig.add_subplot(gs[0, 0], projection=map_proj)
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0], projection=map_proj)
ax4 = fig.add_subplot(gs[1, 1])

# 颜色等级（非等间距分级）
levels_list = [
    [0, 10, 25, 45, 70, 100, 125, 150],       # 第一个地图，指定非等距分级
    [0, 50, 100,200,300, 500, 750, 1000,1500],          # 第二个地图
]

# 标题列表
titles = [
    "Valid MLD sample count",
    "Profile count of all",
]

cmap = plt.cm.viridis  # 多色渐变colormap

def wrap_global_longitude(data):
    """将经度及其对应数据重排到 [30, 390)，并补循环列消除接缝。"""
    wrapped_lon = ((data['lon'] - 30) % 360) + 30
    data_wrapped = data.assign_coords(lon=wrapped_lon).sortby('lon')
    cyclic = data_wrapped.isel(lon=0).assign_coords(
        lon=data_wrapped['lon'].isel(lon=0) + 360
    )
    return xr.concat([data_wrapped, cyclic], dim='lon')


def plot_map_with_colorbar(ax, data, label_letter, levels, title):
    # 色标缩短并居中，同时与地图下边界留出距离
    cax = inset_axes(
        ax, width='78%', height='4.5%', loc='lower center',
        bbox_to_anchor=(0, -0.14, 1, 1), bbox_transform=ax.transAxes,
        borderpad=0,
    )

    # 过滤陆地，将非海洋点设为NaN，避免被绘制为色阶0
    data_plot = wrap_global_longitude(data.where(data > 0))

    im = data_plot.plot.contourf(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        add_colorbar=False,
        extend='max',
        zorder=1,
    )
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=levels)
    cbar.set_label(' ', fontsize=15)
    cbar.ax.tick_params(labelsize=16)

    ax.set_title(title, fontsize=21, pad=10)
    ax.coastlines(linewidth=0.8, zorder=2)  # 线条层放在陆地之上

    # **确保陆地图层在最顶层**
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=3)

    ax.set_global()

    # Robinson 投影中心为 210°E，外侧接缝对应 30°E/390°E
    meridians = [30, 90, 150, -150, -90, -30]
    y_positions = np.arange(-60, 61, 30)
    ax.set_xlabel('')
    ax.set_ylabel('')
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(), xlocs=meridians, ylocs=y_positions,
        draw_labels=True, linewidth=0.7, color='gray', alpha=0.65, zorder=4,
        x_inline=False, y_inline=False,
    )
    gl.top_labels = False
    gl.bottom_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.xlabel_style = {'size': 16, 'family': 'Times New Roman'}
    gl.ylabel_style = {'size': 16, 'family': 'Times New Roman'}

    # 非矩形投影仅保留中间三个连续经度标签
    for x_frac, lon_label in zip([1/3, 1/2, 2/3], [150, 210, 270]):
        ax.text(
            x_frac, -0.015, f'{lon_label}°E', transform=ax.transAxes,
            fontsize=16, fontfamily='Times New Roman', ha='center', va='top',
            clip_on=False,
        )

    ax.text(0.02, 0.98, f'({label_letter})', transform=ax.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='left',
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))


def plot_line(ax, years, values, label_letter, ylabel, color):
    ax.plot(years, values, color=color, linewidth=4, marker='o', markersize=4)
    ax.set_xlabel('Date (YYYY)', fontsize=22)
    ax.set_ylabel(ylabel, fontsize=24)
    ax.set_xticks(np.arange(1985, 2025, 5))
    ax.set_xlim(1981.0, 2024 + 9/12)  # 1981年1月至2024年10月初
    ax.tick_params(axis='both', labelsize=21)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.yaxis.set_major_formatter(ticker.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.yaxis.get_offset_text().set_fontsize(21)
    ax.text(0.035, 0.93, f'({label_letter})', transform=ax.transAxes,
            fontsize=22, fontweight='bold', va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

# 绘制地图
plot_map_with_colorbar(ax1, plot_data1, 'a', levels_list[0], titles[0])
plot_map_with_colorbar(ax3, plot_data2, 'c', levels_list[1], titles[1])

# 绘制折线图
plot_line(ax2, years1, values1, 'b', 'Annual data count', 'green')
plot_line(ax4, years2, values2, 'd', 'Annual profile count', 'green')

fig.subplots_adjust(left=0.055, right=0.985, top=0.95, bottom=0.085)

# B、D 整体稍向下移，使其视觉中心与左侧地图及色标组合对齐
for ax in (ax2, ax4):
    pos = ax.get_position()
    ax.set_position([pos.x0, pos.y0 - 0.012, pos.width, pos.height + 0.025])

# 以600 dpi导出最终图，紧边界且不保留额外白边
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(
    output_dir / 's1figure.png',
    dpi=600,
    bbox_inches='tight',
    pad_inches=0,
    facecolor='white',
)
plt.show()
